# Bloque 1. Panorama de formatos de datos en ML

**Módulo ETCDFM (5104) · UD2: Formatos de fichero de datos**

---

Elegir el formato de fichero adecuado puede marcar la diferencia entre un pipeline lento y costoso y uno eficiente y mantenible. En este bloque hacemos un recorrido general por los formatos más habituales: su clasificación, características, criterios de elección y tendencias actuales en el sector.


## Contenidos

1. Clasificación de formatos: texto vs binario; orientación por filas vs columnar
2. Tabla comparativa de formatos para ML
3. Demostración: tamaño de fichero según formato
4. Criterios de elección de formato
5. Formatos y ubicación: local, nube y sistemas distribuidos
6. Tendencias actuales


## 1.1 Clasificación de formatos

Los formatos de datos se clasifican desde dos perspectivas:

**Por codificación:**
- **Texto plano** — legibles con cualquier editor. Portables e inspeccionables, pero más lentos y voluminosos.
- **Binarios** — no legibles directamente, pero más compactos y rápidos de leer/escribir.

**Por orientación de almacenamiento:**
- **Por filas** (*row-oriented*) — todos los campos de un registro se almacenan juntos. Óptimo para insertar o leer registros completos (OLTP).
- **Por columnas** (*columnar*) — todos los valores de un campo se almacenan juntos. Óptimo para análisis y ML: se leen solo las columnas necesarias, y la compresión es más eficiente.


In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch

fig, ax = plt.subplots(figsize=(14, 7))
ax.set_xlim(0, 14)
ax.set_ylim(0, 7)
ax.axis('off')

def box(ax, x, y, w, h, label, color, fs=11, bold=True):
    patch = FancyBboxPatch((x - w/2, y - h/2), w, h,
                           boxstyle='round,pad=0.2', linewidth=1.5,
                           edgecolor='#37474F', facecolor=color, zorder=3)
    ax.add_patch(patch)
    fw = 'bold' if bold else 'normal'
    ax.text(x, y, label, ha='center', va='center', fontsize=fs, fontweight=fw, zorder=4)

def edge(ax, x1, y1, x2, y2):
    ax.plot([x1, x2], [y1, y2], color='#78909C', linewidth=1.5, zorder=1)

# Raiz
box(ax, 7, 6.4, 3.4, 0.7, 'Formatos de datos', '#B0BEC5', fs=13)

# Nivel 1
box(ax, 3, 5.1, 3.2, 0.7, 'Texto plano', '#90CAF9', fs=12)
box(ax, 11, 5.1, 3.2, 0.7, 'Binario', '#A5D6A7', fs=12)
edge(ax, 7, 6.05, 3, 5.45)
edge(ax, 7, 6.05, 11, 5.45)

# Hijos texto plano
tp = [('CSV / TSV', 1.2, 3.8), ('XML', 3.0, 3.8), ('JSON / NDJSON', 5.0, 3.8)]
for lbl, x, y in tp:
    box(ax, x, y, 1.6, 0.6, lbl, '#E3F2FD', fs=10)
    edge(ax, 3, 4.75, x, 4.1)

# Nivel 2 binario
box(ax, 9.2, 3.8, 2.8, 0.65, 'Por filas', '#C8E6C9', fs=11)
box(ax, 12.8, 3.8, 2.8, 0.65, 'Por columnas', '#C8E6C9', fs=11)
edge(ax, 11, 4.75, 9.2, 4.12)
edge(ax, 11, 4.75, 12.8, 4.12)

# Hijos filas
fr = [('Excel', 7.8, 2.5), ('Avro', 9.2, 2.5), ('Protobuf', 10.6, 2.5)]
for lbl, x, y in fr:
    box(ax, x, y, 1.3, 0.55, lbl, '#F1F8E9', fs=10)
    edge(ax, 9.2, 3.47, x, 2.78)

# Hijos columnar
cl = [('Parquet', 11.9, 2.5), ('HDF5', 13.1, 2.5), ('Arrow/Feather', 12.5, 1.5)]
for lbl, x, y in cl:
    box(ax, x, y, 1.5, 0.55, lbl, '#F1F8E9', fs=10)
    edge(ax, 12.8, 3.47, x, 2.78 if y > 2 else 1.78)

plt.title('Taxonomia de formatos de datos', fontsize=14, fontweight='bold', pad=10)
plt.tight_layout()
plt.show()


## Orientacion por filas vs por columnas

La **orientación de almacenamiento** determina cómo se distribuyen físicamente los datos en disco.

| | Por filas | Por columnas |
|---|---|---|
| Almacenamiento | Todos los campos de un registro juntos | Todos los valores de un campo juntos |
| Lectura de una fila | Muy rápida | Lenta (hay que reconstruirla) |
| Lectura de una columna | Lenta (hay que saltar) | Muy rápida |
| Compresión | Moderada | Alta (valores similares juntos) |
| Ideal para | Inserción/consulta de registros (OLTP) | Análisis, agregaciones, ML (OLAP) |
| Ejemplos | CSV, Excel, Avro, Protobuf | Parquet, HDF5, Arrow/Feather |


In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
cols_hdr = ['ID', 'Nombre', 'Edad', 'Salario']
data_rows = [[1, 'Ana', 30, 28000], [2, 'Luis', 25, 24000], [3, 'Eva', 35, 32000]]
row_clrs = ['#BBDEFB', '#C8E6C9', '#FFF9C4']
col_clrs = ['#BBDEFB', '#FFE0B2', '#F8BBD0', '#E1BEE7']

def header_box(ax, x, y, w, h, label):
    p = FancyBboxPatch((x - w/2, y - h/2), w, h,
                       boxstyle='round,pad=0.05', lw=1.5,
                       edgecolor='#455A64', facecolor='#90A4AE', zorder=3)
    ax.add_patch(p)
    ax.text(x, y, label, ha='center', va='center', fontsize=11, fontweight='bold', zorder=4)

def data_box(ax, x, y, w, h, val, color):
    p = FancyBboxPatch((x - w/2, y - h/2), w, h,
                       boxstyle='round,pad=0.05', lw=1.2,
                       edgecolor='#455A64', facecolor=color, zorder=3)
    ax.add_patch(p)
    ax.text(x, y, str(val), ha='center', va='center', fontsize=11, zorder=4)

CW, CH, RH = 1.0, 0.52, 0.72  # col width, col height, row height

for idx, (ax, title, groupby) in enumerate([
    (axes[0], 'Por filas  (CSV, Excel, Avro)', 'row'),
    (axes[1], 'Por columnas  (Parquet, HDF5, Arrow)', 'col')
]):
    ax.set_xlim(-0.1, 4.3)
    ax.set_ylim(-0.2, 4.3)
    ax.axis('off')
    ax.set_title(title, fontsize=12, fontweight='bold', pad=6)
    for ci, col in enumerate(cols_hdr):
        header_box(ax, ci * CW + CW/2, 3.9, CW - 0.06, CH, col)
    for ri, row in enumerate(data_rows):
        for ci, val in enumerate(row):
            color = row_clrs[ri] if groupby == 'row' else col_clrs[ci]
            data_box(ax, ci * CW + CW/2, 3.9 - (ri + 1) * RH, CW - 0.06, RH - 0.08, val, color)
    if groupby == 'row':
        for ri in range(3):
            y = 3.9 - (ri + 1) * RH
            ax.annotate('', xy=(4.25, y), xytext=(4.08, y),
                        arrowprops=dict(arrowstyle='->', color=row_clrs[ri], lw=2))
            ax.text(4.28, y, f'fila {ri+1}', fontsize=9, color='#37474F', va='center')
    else:
        for ci in range(4):
            x = ci * CW + CW/2
            ax.annotate('', xy=(x, -0.12), xytext=(x, 3.55),
                        arrowprops=dict(arrowstyle='->', color=col_clrs[ci], lw=2))
        ax.text(2.0, -0.18, 'lectura por columna: muy rapida', ha='center',
                fontsize=10, color='#37474F', style='italic')

plt.suptitle('Orientacion del almacenamiento en disco', fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()


## 1.2 Tabla comparativa de formatos

A continuación se comparan los formatos más habituales en proyectos de ML según sus características clave:
codificación, orientación, soporte de esquema, compresión nativa, compatibilidad con streaming y uso típico.


In [ ]:
import matplotlib.pyplot as plt

formatos = ['CSV', 'Excel', 'XML', 'JSON', 'NDJSON',
            'Parquet', 'HDF5', 'Arrow/Feather', 'Avro', 'Protobuf']
cols_hdr = ['Tipo', 'Orientacion', 'Esquema', 'Compresion', 'Streaming', 'Uso tipico en ML']
table_data = [
    ['Texto',   'Filas',      'No',              'No',  'Si',  'Datos tabulares sencillos'],
    ['Binario', 'Filas',      'Parcial',          'Si',  'No',  'Informes, datos de negocio'],
    ['Texto',   'Jerarquico', 'Si (XSD)',         'No',  'No',  'Intercambio de datos'],
    ['Texto',   'Jerarquico', 'Si (JSON Schema)', 'No',  'No',  'APIs REST, configuracion'],
    ['Texto',   'Filas',      'No',              'No',  'Si',  'Logs, eventos en tiempo real'],
    ['Binario', 'Columnar',   'Si',              'Si',  'No',  'Data lake, entrenamiento ML'],
    ['Binario', 'Columnar',   'Si',              'Si',  'No',  'Arrays numericos grandes, DL'],
    ['Binario', 'Columnar',   'Si',              'Si',  'Si',  'Transferencia en memoria, IPC'],
    ['Binario', 'Filas',      'Si',              'Si',  'Si',  'Kafka, Hadoop, pipelines'],
    ['Binario', 'Filas',      'Si',              'Si',  'Si',  'TensorFlow, microservicios'],
]
clr = {'Texto': '#E3F2FD', 'Binario': '#E8F5E9'}
cell_colors = [[clr[r[0]]] * len(cols_hdr) for r in table_data]

fig, ax = plt.subplots(figsize=(16, 5.5))
ax.axis('off')
tbl = ax.table(
    cellText=table_data, rowLabels=formatos, colLabels=cols_hdr,
    cellColours=cell_colors,
    rowColours=['#CFD8DC'] * len(formatos),
    colColours=['#546E7A'] * len(cols_hdr),
    cellLoc='center', loc='center'
)
tbl.auto_set_font_size(False)
tbl.set_fontsize(11)
tbl.scale(1, 1.55)
for (r, c), cell in tbl.get_celld().items():
    if r == 0:
        cell.set_text_props(color='white', fontweight='bold')
    if c == -1 and r > 0:
        cell.set_text_props(fontweight='bold')
plt.title('Comparativa de formatos de datos para ML', fontsize=14, fontweight='bold', pad=14)
plt.tight_layout()
plt.show()


## Demo: tamaño de fichero según formato

Una forma directa de ver la diferencia entre formatos es guardar el mismo dataset en varios formatos y comparar el espacio ocupado en disco.

El siguiente código crea un dataset sintético de 100 000 filas y lo guarda en CSV, JSON, Excel y Parquet, midiendo el tamaño resultante.


In [ ]:
import pandas as pd
import numpy as np
import os, tempfile, matplotlib.pyplot as plt

# Dataset sintetico
np.random.seed(42)
N = 100_000
df = pd.DataFrame({
    'id':        range(N),
    'nombre':    [f'item_{i:06d}' for i in range(N)],
    'valor':     np.random.randn(N),
    'categoria': np.random.choice(['A', 'B', 'C', 'D'], N),
    'precio':    np.random.uniform(1, 1000, N).round(2),
})
print(f'Dataset: {len(df):,} filas x {len(df.columns)} columnas')

# Guardar en varios formatos
tmpdir = tempfile.mkdtemp()
df.to_csv(f'{tmpdir}/data.csv', index=False)
df.to_json(f'{tmpdir}/data.json', orient='records')
df.to_parquet(f'{tmpdir}/data.parquet')
df.to_parquet(f'{tmpdir}/data_snappy.parquet', compression='snappy')
df.to_feather(f'{tmpdir}/data.feather')

# Comparar tamanos
formatos_demo = {
    'CSV':               f'{tmpdir}/data.csv',
    'JSON':              f'{tmpdir}/data.json',
    'Parquet (gzip)':    f'{tmpdir}/data.parquet',
    'Parquet (snappy)':  f'{tmpdir}/data_snappy.parquet',
    'Feather (Arrow)':   f'{tmpdir}/data.feather',
}
sizes_kb = {k: os.path.getsize(v) / 1024 for k, v in formatos_demo.items()}

colors = ['#90CAF9', '#FFCC80', '#A5D6A7', '#C8E6C9', '#DCEDC8']
fig, ax = plt.subplots(figsize=(10, 4))
bars = ax.barh(list(sizes_kb.keys()), list(sizes_kb.values()), color=colors, edgecolor='#37474F')
ax.set_xlabel('Tamano en disco (KB)', fontsize=12)
ax.set_title(f'Tamano en disco: {N:,} filas, mismo dataset', fontsize=13, fontweight='bold')
for bar, val in zip(bars, sizes_kb.values()):
    ax.text(val + 30, bar.get_y() + bar.get_height()/2,
            f'{val:,.0f} KB', va='center', fontsize=11)
plt.tight_layout()
plt.show()


## 1.3 Criterios de elección de formato

No existe un formato universalmente mejor. La elección depende de:

| Criterio | Pregunta clave | Implicacion |
|---|---|---|
| **Tamaño** | ¿El dataset es mayor de 1 GB? | Binario columnar (Parquet, HDF5) |
| **Velocidad** | ¿Es crítica la velocidad de lectura analítica? | Parquet, Arrow |
| **Interoperabilidad** | ¿Lo leerán otras herramientas/equipos? | CSV, JSON |
| **Esquema** | ¿Necesito garantizar los tipos de datos? | Parquet, Avro, Protobuf |
| **Streaming** | ¿Los datos llegan registro a registro? | NDJSON, Avro, Protobuf |
| **Herramientas ML** | ¿Qué framework voy a usar? | Parquet (sklearn, XGBoost), HDF5 (Keras) |
| **Nube** | ¿Los datos están en S3 / GCS / Azure? | Parquet con particionado |


In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch

fig, ax = plt.subplots(figsize=(14, 5))
ax.set_xlim(0, 14)
ax.set_ylim(0, 5)
ax.axis('off')

def box(ax, x, y, w, h, top, bot, clr, fs=10):
    p = FancyBboxPatch((x - w/2, y - h/2), w, h,
                       boxstyle='round,pad=0.2', lw=1.5,
                       edgecolor='#37474F', facecolor=clr, zorder=3)
    ax.add_patch(p)
    ax.text(x, y + 0.22, top, ha='center', va='center',
            fontsize=fs, fontweight='bold', zorder=4)
    ax.text(x, y - 0.22, bot, ha='center', va='center',
            fontsize=9, zorder=4, style='italic')

criterios = [
    (1.8,  3.7, 2.8, 1.4, 'Tamano del dataset',   '> 1 GB: Parquet / HDF5',  '#BBDEFB'),
    (5.3,  3.7, 2.8, 1.4, 'Velocidad analitica',  'Parquet, Arrow',           '#C8E6C9'),
    (8.8,  3.7, 2.8, 1.4, 'Interoperabilidad',    'CSV, JSON',                '#FFF9C4'),
    (12.2, 3.7, 2.8, 1.4, 'Soporte de esquema',   'Parquet, Avro, Protobuf',  '#FFE0B2'),
    (1.8,  1.5, 2.8, 1.4, 'Streaming',            'NDJSON, Avro, Protobuf',   '#F8BBD0'),
    (5.3,  1.5, 2.8, 1.4, 'Herramientas ML',      'Parquet (sklearn) HDF5 (DL)', '#E1BEE7'),
    (8.8,  1.5, 2.8, 1.4, 'Almacenamiento nube',  'Parquet particionado',     '#B2DFDB'),
    (12.2, 1.5, 2.8, 1.4, 'Lectura por columna',  'Evitar CSV para analisis', '#DCEDC8'),
]
for args in criterios:
    box(ax, *args)

ax.text(7, 4.7, 'Criterios clave para elegir formato de datos',
        ha='center', va='center', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()


## 1.4 Formatos y ubicación de los datos

Los datos no siempre están en el disco local. Según el entorno, el formato condiciona dónde y cómo se pueden usar:

| Ubicacion | Formatos habituales | Herramientas de acceso |
|---|---|---|
| **Local** | CSV, Excel, JSON, Parquet, HDF5 | pandas, h5py, pyarrow |
| **Nube** (S3, GCS, Azure) | Parquet, CSV, NDJSON | pyarrow + s3fs / gcsfs / adlfs |
| **HDFS / Spark** | Parquet, Avro, ORC | PySpark, Hive |
| **Streams** (Kafka, Pub/Sub) | Avro, Protobuf, NDJSON | confluent-kafka, fastavro |

La tendencia en la industria es almacenar en **Parquet sobre almacenamiento objeto en la nube** y leerlo directamente sin descarga completa, gracias al *filter pushdown* y el *projection pushdown* de pyarrow.


In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch

fig, ax = plt.subplots(figsize=(14, 4))
ax.set_xlim(0, 14)
ax.set_ylim(0, 4)
ax.axis('off')

locs = [
    (1.8,  2.0, 3.0, 2.2, 'Local',              'CSV, Excel, JSON\nParquet, HDF5',         '#BBDEFB'),
    (5.5,  2.0, 3.0, 2.2, 'Nube (S3/GCS/Azure)','Parquet, CSV, NDJSON\ns3fs / gcsfs',      '#C8E6C9'),
    (9.2,  2.0, 3.0, 2.2, 'HDFS / Spark',       'Parquet, Avro, ORC\nPySpark, Hive',       '#FFF9C4'),
    (12.5, 2.0, 2.5, 2.2, 'Kafka / Streams',    'Avro, Protobuf\nNDJSON',                  '#FFE0B2'),
]
for x, y, w, h, title, sub, clr in locs:
    p = FancyBboxPatch((x - w/2, y - h/2), w, h,
                       boxstyle='round,pad=0.25', lw=1.5,
                       edgecolor='#37474F', facecolor=clr, zorder=3)
    ax.add_patch(p)
    ax.text(x, y + 0.45, title, ha='center', va='center',
            fontsize=12, fontweight='bold', zorder=4)
    for li, line in enumerate(sub.split('\n')):
        ax.text(x, y - 0.1 - li * 0.38, line, ha='center', va='center',
                fontsize=10, zorder=4)

# Flechas
for x1, x2 in [(3.3, 4.0), (7.0, 7.7), (10.7, 11.25)]:
    ax.annotate('', xy=(x2, 2.0), xytext=(x1, 2.0),
                arrowprops=dict(arrowstyle='->', color='#78909C', lw=1.8))

ax.text(7.15, 3.7, 'Formatos segun ubicacion de los datos',
        ha='center', va='center', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()


## 1.5 Tendencias actuales

**Parquet como estándar de data lake**
Los sistemas lakehouse (Delta Lake, Apache Iceberg, Apache Hudi) usan Parquet como base. Es el formato de facto para datasets de entrenamiento a gran escala en la nube.

**Apache Arrow como estándar de transferencia en memoria**
Arrow define una representación columnar en RAM que elimina copias entre herramientas (pandas 2.0, DuckDB, Polars, Spark, R). Feather es Arrow serializado a disco.

**NDJSON en pipelines de ingesta**
Una línea = un registro JSON. Ideal para logs, exports de MongoDB/Elasticsearch y feeds de eventos en la capa de ingesta de pipelines ML.

**El binario supera al texto en producción**
Parquet puede ser 5–20x más pequeño que CSV para el mismo dataset y leerse 10–50x más rápido en consultas analíticas.


In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch

fig, ax = plt.subplots(figsize=(14, 3.8))
ax.set_xlim(0, 14)
ax.set_ylim(0, 3.8)
ax.axis('off')

tendencias = [
    (1.8,  1.9, 3.0, 1.7, 'Parquet',       'Estandar de data lake\nDelta/Iceberg/Hudi', '#B3E5FC'),
    (5.5,  1.9, 3.0, 1.7, 'Apache Arrow',  'Estandar en memoria\nFeather en disco',     '#DCEDC8'),
    (9.2,  1.9, 3.0, 1.7, 'NDJSON',        'Ingesta y streaming\nLogs, MongoDB exports','#FFE0B2'),
    (12.6, 1.9, 2.6, 1.7, 'Binario > CSV', '5-20x menos espacio\n10-50x mas rapido',   '#F8BBD0'),
]
for x, y, w, h, title, sub, clr in tendencias:
    p = FancyBboxPatch((x - w/2, y - h/2), w, h,
                       boxstyle='round,pad=0.25', lw=2,
                       edgecolor='#37474F', facecolor=clr, zorder=3)
    ax.add_patch(p)
    ax.text(x, y + 0.38, title, ha='center', va='center',
            fontsize=12, fontweight='bold', zorder=4)
    for li, line in enumerate(sub.split('\n')):
        ax.text(x, y - 0.08 - li * 0.38, line, ha='center', va='center',
                fontsize=10, zorder=4)

ax.text(7, 3.5, 'Tendencias dominantes en el ecosistema de datos para ML (2025)',
        ha='center', va='center', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()


## Resumen del bloque

- Los formatos se clasifican por **codificación** (texto / binario) y **orientación** (filas / columnar).
- Los formatos **columnares binarios** (Parquet, HDF5, Arrow) son los más eficientes para ML: menos espacio, lectura más rápida, esquema integrado.
- La elección del formato depende del **tamaño**, **velocidad**, **interoperabilidad**, **esquema** y **entorno** de ejecución.
- En la nube, **Parquet** es el estándar; en memoria, **Arrow**; en streaming, **NDJSON / Avro / Protobuf**.
- Los bloques 2 al 5 de esta unidad desarrollan en profundidad cada familia de formatos.
